# MovieLens 32M — frozen selection and final evaluation

Review saved Validation comparisons, freeze explicit artifact choices, then enable final Test inference only deliberately. This notebook never fits or retunes models. All histories stay frozen at external Train; Validation is not appended.


## 1. Environment and paths

Place this repository's readable `src/recsys/`, `configs/` and `tests/` directories in the Google Drive project directory below. No other notebook needs to have run in this runtime. The existing dataset and split artifacts must already exist for real experiments. The source path and file hashes are checked before imports; restart the runtime if a different or edited package is cached. Dependency requirements and installation instructions are in the project README; no automatic package installation runs here.


In [ ]:
from pathlib import Path
import sys
import importlib.util
RUN_MODE = "smoke_test"
if RUN_MODE == "smoke_test":
    import os
    for _thread_variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
        os.environ[_thread_variable] = "1"  # bound synthetic CPU native-library concurrency before imports
SUPPORTED_MODES = ('smoke_test', 'load_evaluate')
if RUN_MODE not in SUPPORTED_MODES:
    raise ValueError(f"Unsupported mode {RUN_MODE!r}; choose {SUPPORTED_MODES}")
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "personal_projects/movielens_hybrid_recsys"
)
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount("/content/drive")
SRC_DIR = PROJECT_DIR / "src"
if not (SRC_DIR / "recsys" / "__init__.py").is_file():
    raise FileNotFoundError(f"Readable source missing: {SRC_DIR / 'recsys'}. Copy or clone the repository there.")
import hashlib
_source_hash = hashlib.sha256()
for _file in sorted((SRC_DIR / "recsys").rglob("*.py")):
    _source_hash.update(str(_file.relative_to(SRC_DIR)).encode())
    _source_hash.update(_file.read_bytes())
_loaded = sys.modules.get("recsys")
if _loaded is not None:
    if (not Path(_loaded.__file__).resolve().is_relative_to(SRC_DIR.resolve()) or
        getattr(_loaded, "__notebook_source_hash__", None) != _source_hash.hexdigest()):
        raise RuntimeError("A stale/different recsys module is cached. Restart this runtime before importing.")
sys.path = [str(SRC_DIR)] + [p for p in sys.path if p != str(SRC_DIR) and "recsys_source" not in p]
import recsys
recsys.__notebook_source_hash__ = _source_hash.hexdigest()
from recsys.runtime import environment
from recsys.smoke import run_smoke
print("Source:", recsys.__file__)
print(environment("/content" if Path("/content").exists() else "/tmp"))


In [ ]:
if RUN_MODE == "smoke_test":
    print(run_smoke(PROJECT_DIR, 'final'))


## 2. Explicit selection inputs and Test switches

Use the exact paths printed by notebooks 03–09. `FREEZE_SELECTION` seals a Validation-only choice. Merely loading this notebook or choosing `load_evaluate` does not evaluate Test. Enabling Test also requires the exact sealed selection ID.


In [ ]:
from recsys.contracts import CHANNELS
from recsys.final_evaluation import freeze_selection, load_selection, evaluate_selected
import json
VALIDATION_CANDIDATES = {source: None for source in CHANNELS}
VALIDATION_FUSION = None
RANKER_CHECKPOINT = None
RANKER_VALIDATION = None
FREEZE_SELECTION = False
SELECTION_PATH = None
RUN_FINAL_TEST = False
CONFIRM_SELECTION_ID = ""
FINAL_OUTPUT = None


## 3. Freeze matching Validation artifacts

All six sources must share the same split, full-Train cutoff, catalog, mappings, users/labels and budget. Fusion must reference those exact candidates, and the selected ranker must have used that fusion for Validation. No latest-run guessing or Test-based model selection occurs.


In [ ]:
if RUN_MODE != "smoke_test" and FREEZE_SELECTION:
    if (any(p is None for p in VALIDATION_CANDIDATES.values()) or VALIDATION_FUSION is None or
        RANKER_CHECKPOINT is None or RANKER_VALIDATION is None):
        raise ValueError("Set all explicit Validation selection artifact paths.")
    SELECTION_PATH = freeze_selection(PROJECT_DIR, VALIDATION_CANDIDATES, VALIDATION_FUSION,
                                      RANKER_CHECKPOINT, RANKER_VALIDATION)
    print("Frozen selection:", SELECTION_PATH)


## 4. Load saved selections and previous Validation results

This step verifies immutable references and displays already computed results without retraining or reading Test labels. Models can be trained in different Colab runtimes. Their source hashes and contracts identify their provenance.


In [ ]:
import pandas as pd
from IPython.display import display
if RUN_MODE != "smoke_test":
    if SELECTION_PATH is None:
        raise ValueError("Set an existing selection path, or freeze explicit Validation inputs above.")
    selection = load_selection(SELECTION_PATH)
    print("Selection ID:", selection["selection_id"])
    _fusion_path = Path(selection["validation_fusion"]["path"])
    _validation = json.loads((_fusion_path / "metrics.json").read_text())
    display(pd.DataFrame([{**reports[0], "model": name} for name, reports in _validation["metrics"].items()]))
    _ranker_result = Path(selection["ranker_validation"]["path"])
    display(pd.DataFrame(json.loads((_ranker_result / "metrics.json").read_text())["metrics"]["lightgbm"]))


## 5. Explicitly gated Test inference

Only this step deserializes Test labels. It loads each selected full-Train checkpoint, writes standardized Test candidates, fuses them and scores the selected LambdaMART model. No fit or tuning call occurs. Completed Test results are reused after checksum verification. Source/data incompatibilities fail clearly.


In [ ]:
if RUN_MODE != "smoke_test" and RUN_FINAL_TEST:
    FINAL_OUTPUT = evaluate_selected(PROJECT_DIR, SELECTION_PATH, enabled=True,
                                     confirmation=CONFIRM_SELECTION_ID)
    print("Final result:", FINAL_OUTPUT)
elif RUN_MODE != "smoke_test":
    print("Test is disabled; only saved Validation results were reviewed.")


## 6. Final comparisons, contributions and resources

Individual channels, candidate union ceiling, RRF, LambdaMART and every remove-one-channel ablation use the same frozen Test population. Report Recall@20/50/100, NDCG@20, HitRate, coverage, popularity concentration, user-bootstrap intervals, training time, sampled CPU RSS and GPU allocator peaks. Conditional ranking metrics are separately labeled.


In [ ]:
if FINAL_OUTPUT is not None:
    final = json.loads((FINAL_OUTPUT / "metrics.json").read_text())
    display(pd.DataFrame([{**reports[0], "model": name,
                           "catalog_coverage": final["catalog_coverage"][name],
                           "popular_top_decile_share": final["top_decile_popularity_share"][name]}
                          for name, reports in final["metrics"].items()]))
    display(pd.DataFrame(final["conditional_on_pool_hit"]["lightgbm"]))
    display(pd.Series(final["unique_relevant_interactions"], name="unique positives"))
    _resources = {name: {**info["training_profile"], "checkpoint_bytes": info["checkpoint_bytes"],
                         "retrieval_seconds": info["evaluation"]["retrieval_seconds"]}
                  for name, info in final["profiles"].items()}
    _resources["lightgbm"] = final["ranker_training"]["profile"]
    display(pd.DataFrame(_resources).T)
    print("Pool recall ceiling:", final["union_recall_ceiling"])
    print("Excluded positive labels:", final["eligibility"])


## 7. Interpretation and experiment record

Test scores are final estimates for the sealed selection, never tuning signals. Rating timestamps are submission times, not consumption or exposure. Missing exposure logs, unknown-pair supervision, MNAR feedback, bursts and temporal shifts prevent online or causal claims. Candidate pool recall limits ranking. Historical/full-Train feature mismatch is documented in the saved ranker sampling report. No real benchmark numbers were generated during this refactor.
